# Build train hotspot features

Transform cleaned trip-level occupancy into station and time-bucket hotspot signals.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.append(str(PROJECT_ROOT / 'src'))

from hotspot_features import add_flow_features, add_time_features, aggregate_station_hotspots


In [ ]:
clean_path = PROJECT_ROOT / 'data' / 'interim' / 'train_2018_clean.parquet'
df_clean = pd.read_parquet(clean_path)
df_flow = add_flow_features(df_clean, trip_col='trip_id', seq_col='stop_sequence', occupancy_col='occupancy_numeric')
df_flow = add_time_features(df_flow, time_col='event_time')
df_flow.head()

In [ ]:
df_hotspots = aggregate_station_hotspots(
    df_flow,
    station_col='station_name',
    lat_col='latitude' if 'latitude' in df_flow.columns else None,
    lon_col='longitude' if 'longitude' in df_flow.columns else None,
)
print(df_hotspots.shape)
df_hotspots.head()

In [ ]:
output_path = PROJECT_ROOT / 'data' / 'processed' / 'train_hotspots_2018.parquet'
df_hotspots.to_parquet(output_path, index=False)
print(f'Saved hotspot parquet to {output_path}')